In [1]:
import os
import shutil
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits

# Importar MontagePy
try:
    from MontagePy.main import mHdr, mImgtbl, mProjExec, mAdd
    MONTAGE_AVAILABLE = True
    print("✅ MontagePy está disponible")
except ImportError:
    MONTAGE_AVAILABLE = False
    print("❌ MontagePy no está instalado. Ejecuta: pip install MontagePy")

✅ MontagePy está disponible


In [2]:
def crear_mosaico_y_rgb_comprobado():
    """
    Crea mosaicos con Montage y luego RGB con tu método comprobado
    """
    print("🛠️ CREANDO MOSAICOS CON MONTAGE + RGB CON MÉTODO COMPROBADO")
    print("=" * 60)
    
    # Configuración
    work_dir = os.path.abspath("../anac_data/montage_work")
    output_dir = os.path.abspath("../anac_data/Figs-images")
    data_base_dir = os.path.abspath("../anac_data")
    
    # Campos a procesar (24 campos)
    campos = [f'CenA{i:02d}' for i in range(1, 25)]
    
    print(f"📁 Procesando {len(campos)} campos...")
    
    # Limpiar directorio de trabajo
    if os.path.exists(work_dir):
        shutil.rmtree(work_dir)
    
    os.makedirs(work_dir)
    os.makedirs(f"{work_dir}/raw", exist_ok=True)
    
    # Guardar directorio original
    original_dir = os.getcwd()
    os.chdir(work_dir)
    
    try:
        # PRIMERO: Crear mosaicos con Montage (esto ya funciona)
        mosaicos_montage = {}
        
        for filtro in ['F861', 'F660', 'F515']:
            print(f"\n🌈 PROCESANDO FILTRO {filtro}...")
            
            # Preparar datos del filtro
            archivos_filtro = []
            for campo in campos:
                src_path = f"{data_base_dir}/{campo}/{campo}_{filtro}.fits.fz"
                dst_path = f"raw/{campo}_{filtro}.fits"
                
                if os.path.exists(src_path):
                    try:
                        with fits.open(src_path) as hdul:
                            if len(hdul) > 1:
                                data = hdul[1].data
                                header = hdul[1].header
                            else:
                                data = hdul[0].data
                                header = hdul[0].header
                            
                            primary_hdu = fits.PrimaryHDU(data=data, header=header)
                            primary_hdu.writeto(dst_path, overwrite=True)
                            archivos_filtro.append(dst_path)
                        print(f"   ✅ {campo}_{filtro}")
                    except Exception as e:
                        print(f"   ❌ Error: {e}")
            
            if not archivos_filtro:
                continue
            
            # Directorios específicos
            raw_filtro_dir = f"raw_{filtro}"
            projected_filtro_dir = f"projected_{filtro}"
            os.makedirs(raw_filtro_dir, exist_ok=True)
            os.makedirs(projected_filtro_dir, exist_ok=True)
            
            # Copiar archivos
            for archivo in archivos_filtro:
                shutil.copy(archivo, f"{raw_filtro_dir}/{os.path.basename(archivo)}")
            
            # Header de referencia (usando valores que sabemos funcionan)
            ra_center = 201.3651
            dec_center = -43.0191
            size = 8.0
            
            rtn = mHdr(f"{ra_center} {dec_center}", size, size, f"region_{filtro}.hdr")
            if rtn['status'] != '0':
                crear_header_simple(ra_center, dec_center, size, f"region_{filtro}.hdr")
            
            # Procesar con Montage
            rtn_tbl = mImgtbl(raw_filtro_dir, f"rimages_{filtro}.tbl")
            if rtn_tbl['status'] != '0':
                continue
            
            rtn_proj = mProjExec(raw_filtro_dir, f"rimages_{filtro}.tbl", f"region_{filtro}.hdr", 
                                projdir=projected_filtro_dir, quickMode=True)
            if rtn_proj['status'] != '0':
                continue
            
            rtn_ptbl = mImgtbl(projected_filtro_dir, f"pimages_{filtro}.tbl")
            if rtn_ptbl['status'] != '0':
                continue
            
            # Crear mosaico
            mosaic_file = f"mosaic_montage_{filtro}.fits"
            rtn_add = mAdd(projected_filtro_dir, f"pimages_{filtro}.tbl", f"region_{filtro}.hdr", mosaic_file)
            
            if rtn_add['status'] == '0':
                mosaicos_montage[filtro] = mosaic_file
                print(f"✅ Mosaico {filtro} creado")
            
            # Limpiar
            try:
                shutil.rmtree(raw_filtro_dir)
                shutil.rmtree(projected_filtro_dir)
            except:
                pass
        
        # SEGUNDO: Usar TU método comprobado para crear el RGB
        if len(mosaicos_montage) == 3:
            print("\n🌈 CREANDO RGB CON TU MÉTODO COMPROBADO...")
            resultado = crear_rgb_con_metodo_comprobado(mosaicos_montage, output_dir, campos)
            
            if resultado:
                # Guardar mosaicos
                for filtro, archivo in mosaicos_montage.items():
                    shutil.copy(archivo, f"{output_dir}/mosaic_{filtro}_montage.fits")
                    print(f"✅ Mosaico {filtro} guardado")
                
                return mosaicos_montage
        
        return None
            
    except Exception as e:
        print(f"\n❌ Error: {e}")
        import traceback
        traceback.print_exc()
        return None
        
    finally:
        os.chdir(original_dir)

def crear_header_simple(ra, dec, size, output_file):
    """Header simple"""
    naxis = 2000
    
    header_content = f"""SIMPLE  =                    T / file does conform to FITS standard
BITPIX  =                  -64 / number of bits per data pixel
NAXIS   =                    2 / number of data axes
NAXIS1  =                {naxis} / length of data axis 1
NAXIS2  =                {naxis} / length of data axis 2
EXTEND  =                    T / FITS dataset may contain extensions
CTYPE1  = 'RA---TAN'           / Right Ascension, gnomonic projection
CTYPE2  = 'DEC--TAN'           / Declination, gnomonic projection
CRVAL1  = {ra:20.10f} / [deg] Reference coordinate on axis 1
CRVAL2  = {dec:20.10f} / [deg] Reference coordinate on axis 2
CRPIX1  =              {naxis/2:.1f} / [pixel] Reference pixel on axis 1
CRPIX2  =              {naxis/2:.1f} / [pixel] Reference pixel on axis 2
CDELT1  = {-(size/naxis):20.10f} / [deg/pixel] Coordinate increment
CDELT2  = { (size/naxis):20.10f} / [deg/pixel] Coordinate increment
CROTA2  =                  0.0 / [deg] Rotation angle
EQUINOX =               2000.0 / Equinox of celestial coordinate system
"""
    
    with open(output_file, 'w') as f:
        f.write(header_content)

def crear_rgb_con_metodo_comprobado(mosaicos_dict, output_dir, campos):
    """
    Usa TU método comprobado de APLpy pero con los mosaicos de Montage
    """
    try:
        print("🎨 Aplicando tu método comprobado de APLpy a los mosaicos...")
        
        # Crear directorio temporal
        temp_dir = "temp_mosaic_rgb"
        os.makedirs(temp_dir, exist_ok=True)
        
        # Archivos temporales (usando los mosaicos como entrada)
        rgb_cube = f"{temp_dir}/mosaic_rgb_cube.fits"
        rgb_png = f"{temp_dir}/mosaic_rgb_image.png"
        
        # 1. Crear cubo RGB desde los mosaicos de Montage
        print(" - Creando cubo RGB desde mosaicos...")
        import aplpy
        aplpy.make_rgb_cube([
            mosaicos_dict['F861'], 
            mosaicos_dict['F660'], 
            mosaicos_dict['F515']
        ], rgb_cube)
        
        # 2. Crear imagen RGB con ajustes optimizados (TUS parámetros)
        print(" - Generando imagen RGB...")
        aplpy.make_rgb_image(
            rgb_cube, rgb_png,
            stretch_r='log', stretch_g='log', stretch_b='log',
            vmin_r=0.01, vmin_g=0.01, vmin_b=0.01,
            vmax_r=0.95, vmax_g=0.95, vmax_b=0.95
        )
        
        # 3. Mostrar resultado con información (TU formato)
        print(" - Cargando y mostrando resultado...")
        img = plt.imread(rgb_png)
        fig, ax = plt.subplots(figsize=(20, 18), facecolor='black')
        ax.imshow(img)
        ax.axis('off')
        
        # Información detallada del mosaico
        with fits.open(mosaicos_dict['F861']) as hdul:
            header = hdul[0].header
            ra = header.get('CRVAL1', 'N/A')
            dec = header.get('CRVAL2', 'N/A')
            filter_r = 'F861'
        
        with fits.open(mosaicos_dict['F660']) as hdul:
            filter_g = 'F660'
        
        with fits.open(mosaicos_dict['F515']) as hdul:
            filter_b = 'F515'
        
        # Título principal (TU formato)
        plt.title(f'Centaurus A - Mosaico Completo ({len(campos)} campos)\nTelescope: T80/T80Cam - J-PLUS Survey', 
                  color='white', size=22, weight='bold', pad=40)
        
        # Información detallada (TU formato)
        info_text = f'''Coordinates: RA={ra:.6f}°, DEC={dec:.6f}°
Filters: {filter_r}(R), {filter_g}(G), {filter_b}(B)
Fields: {len(campos)} (CenA01 to CenA24)
Method: MontagePy + APLpy
Image Size: {img.shape[1]}×{img.shape[0]} pixels'''

        ax.text(0.02, 0.98, info_text, transform=ax.transAxes, color='white', 
                fontsize=14, verticalalignment='top', family='monospace',
                bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))
        
        # Barra de escala (TU método)
        height = img.shape[0]
        pixel_scale = 0.55  # arcsec/pixel
        scale_arcmin = 30   # Para mosaico grande
        scale_pixels = int(scale_arcmin * 60 / pixel_scale)
        
        ax.plot([50, 50 + scale_pixels], [100, 100], 
                color='yellow', linewidth=8)
        ax.text(50 + scale_pixels/2, 80, f'{scale_arcmin} arcmin', 
                color='yellow', ha='center', va='top', 
                fontsize=18, weight='bold')
        
        # Guardar imagen
        output_path = f"{output_dir}/mosaico_final_comprobado_{len(campos)}campos.png"
        plt.savefig(output_path, dpi=300, bbox_inches='tight', 
                   facecolor='black', edgecolor='none')
        print(f"✅ Imagen guardada: {output_path}")
        
        plt.show()
        
        # 4. Limpiar archivos temporales
        print(" - Limpiando archivos temporales...")
        for temp_file in [rgb_cube, rgb_png]:
            if os.path.exists(temp_file):
                os.remove(temp_file)
        
        try:
            os.rmdir(temp_dir)
        except:
            pass
            
        print(" ✅ ¡Procesamiento RGB completado exitosamente!")
        return True
        
    except Exception as e:
        print(f"❌ Error en método comprobado: {e}")
        import traceback
        traceback.print_exc()
        
        # Si APLpy falla, intentar método alternativo
        return crear_rgb_alternativo(mosaicos_dict, output_dir, campos)

def crear_rgb_alternativo(mosaicos_dict, output_dir, campos):
    """Método alternativo si APLpy falla"""
    try:
        print("🎨 Usando método alternativo con matplotlib...")
        
        # Cargar datos
        data_r = fits.getdata(mosaicos_dict['F861'])
        data_g = fits.getdata(mosaicos_dict['F660']) 
        data_b = fits.getdata(mosaicos_dict['F515'])
        
        print(f"📊 Dimensiones mosaicos: R={data_r.shape}, G={data_g.shape}, B={data_b.shape}")
        
        # Normalización mejorada
        def normalizar_canal(data):
            # Usar percentiles robustos
            data_pos = data[data > 0]
            if len(data_pos) == 0:
                return np.zeros_like(data)
            
            p_low = np.percentile(data_pos, 5)
            p_high = np.percentile(data_pos, 98)
            
            # Transformación logarítmica suave
            data_norm = np.log1p(np.clip(data, p_low, p_high))
            data_norm = (data_norm - np.log1p(p_low)) / (np.log1p(p_high) - np.log1p(p_low))
            return np.clip(data_norm, 0, 1)
        
        r_norm = normalizar_canal(data_r)
        g_norm = normalizar_canal(data_g) 
        b_norm = normalizar_canal(data_b)
        
        # Combinar RGB
        rgb_image = np.stack([r_norm, g_norm, b_norm], axis=-1)
        
        # Crear figura
        fig, ax = plt.subplots(figsize=(18, 16), facecolor='black')
        ax.imshow(rgb_image, origin='lower')
        ax.axis('off')
        
        # Título e información
        ax.set_title(f'Centaurus A - Mosaico Alternativo ({len(campos)} campos)\nF861(R) + F660(G) + F515(B)', 
                    color='white', size=18, pad=20)
        
        # Barra de escala
        height, width = rgb_image.shape[:2]
        pixel_scale = 0.55
        scale_arcmin = 30
        scale_pixels = int(scale_arcmin * 60 / pixel_scale)
        
        bar_y = height * 0.05
        bar_x = width * 0.05
        
        ax.plot([bar_x, bar_x + scale_pixels], [bar_y, bar_y], 
                color='yellow', linewidth=6)
        ax.text(bar_x + scale_pixels/2, bar_y - height*0.02, f'{scale_arcmin} arcmin', 
                color='yellow', ha='center', va='top', fontsize=14, weight='bold')
        
        # Guardar
        output_path = f"{output_dir}/mosaico_alternativo_{len(campos)}campos.png"
        plt.savefig(output_path, dpi=300, bbox_inches='tight', 
                   facecolor='black', edgecolor='none')
        print(f"✅ Imagen alternativa guardada: {output_path}")
        
        plt.show()
        return True
        
    except Exception as e:
        print(f"❌ Error en método alternativo: {e}")
        return False

In [ ]:
# Ejecutar
if __name__ == "__main__":
    print("🚀 EJECUTANDO CON MÉTODO COMPROBADO")
    resultado = crear_mosaico_y_rgb_comprobado()
    
    if resultado:
        print(f"\n🎉 ¡ÉXITO! Mosaico de {len(resultado)} filtros completado")
    else:
        print("\n💥 El proceso no pudo ser completado")

🚀 EJECUTANDO CON MÉTODO COMPROBADO
🛠️ CREANDO MOSAICOS CON MONTAGE + RGB CON MÉTODO COMPROBADO
📁 Procesando 24 campos...

🌈 PROCESANDO FILTRO F861...
   ✅ CenA01_F861
   ✅ CenA02_F861
   ✅ CenA03_F861
   ✅ CenA04_F861
   ✅ CenA05_F861
   ✅ CenA06_F861
   ✅ CenA07_F861
   ✅ CenA08_F861
   ✅ CenA09_F861
   ✅ CenA10_F861
   ✅ CenA11_F861
   ✅ CenA12_F861
   ✅ CenA13_F861
   ✅ CenA14_F861
   ✅ CenA15_F861
   ✅ CenA16_F861
   ✅ CenA17_F861
   ✅ CenA18_F861
   ✅ CenA19_F861
   ✅ CenA20_F861
   ✅ CenA21_F861
   ✅ CenA22_F861
   ✅ CenA23_F861
   ✅ CenA24_F861
✅ Mosaico F861 creado

🌈 PROCESANDO FILTRO F660...
   ✅ CenA01_F660
   ✅ CenA02_F660
   ✅ CenA03_F660
   ✅ CenA04_F660
   ✅ CenA05_F660
   ✅ CenA06_F660
   ✅ CenA07_F660
   ✅ CenA08_F660
   ✅ CenA09_F660
   ✅ CenA10_F660
   ✅ CenA11_F660
   ✅ CenA12_F660
   ✅ CenA13_F660
   ✅ CenA14_F660
   ✅ CenA15_F660
   ✅ CenA16_F660
   ✅ CenA17_F660
   ✅ CenA18_F660
   ✅ CenA19_F660
   ✅ CenA20_F660
   ✅ CenA21_F660
   ✅ CenA22_F660
   ✅ CenA23_F660
